In [42]:
# загружаем библиотеки и данные

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("titanic.csv")
print("Данные загружены", df.shape)
df.head(1)

Данные загружены (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S


In [43]:
# удаление неинформативных и ненужных колонок так как они ничем не помогает для анализа
df = df.drop(columns=["PassengerId", "Ticket", "Name", "Cabin"])
print("Удалены неинформативные ячейки")
df.head(1)

Удалены неинформативные ячейки


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.25,S


In [44]:
# Обработка пропусков медианой или модай что бы Null значения не ломали модель
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print("Пропуски заполнены проверка пустых значений: ", df.isnull().sum())

Пропуски заполнены проверка пустых значений:  Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [45]:
# Компьютер понимает только числа так что кодируем ячейки
df['Sex'] = df['Sex'].replace({'male': 0, 'female': 1})
df['Embarked'] = df['Embarked'].replace({'S': 0, 'C': 1, 'Q': 2})
print("После превращение в числа")
df.head(1)

После превращение в числа


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.25,0


In [46]:
# Разбиение на X(входные данные) и Y на ответ функции
X = df.drop("Survived", axis=1)
y = df["Survived"]

print("Осталось ли пустых значений (NaN)?")
print(X.isnull().sum())

Осталось ли пустых значений (NaN)?
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [47]:
# Разбиваем данные на 80% чтобы учился и на 20% чтобы тестировался
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Данные закодированы и разделены!")
print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")


Данные закодированы и разделены!
Размер обучающей выборки: (712, 7)
Размер тестовой выборки: (179, 7)


In [48]:
# Нормализация данных привести в промежутки как -2 до 2
scaler = StandardScaler()

# Обучаем scaler только на train и применяем к train и test
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Данные успешно нормализованы! (StandardScaler применен)")

Данные успешно нормализованы! (StandardScaler применен)


In [49]:
# Обучение трех классификаций

models = {
    "Логистическая регрессия": LogisticRegression(),
    "Случайный лес (Random Forest)": RandomForestClassifier(n_estimators=100, random_state=42),
    "K-ближайших соседей (KNN)": KNeighborsClassifier(n_neighbors=5)
}

# Словарь для сохранения результатов
results = {}

print("--- НАЧАЛО ОБУЧЕНИЯ ---")
for name, model in models.items():
    # 1. Обучаем модель
    model.fit(X_train, y_train)
    # 2. Делаем предсказание на тестовых данных
    predictions = model.predict(X_test)
    # 3. Вычисляем точность (Accuracy)
    accuracy = accuracy_score(y_test, predictions)
    # Сохраняем в словарь
    results[name] = accuracy
    print(f"Модель '{name}' обучена. Точность: {accuracy*100:.2f}%")

--- НАЧАЛО ОБУЧЕНИЯ ---
Модель 'Логистическая регрессия' обучена. Точность: 79.89%
Модель 'Случайный лес (Random Forest)' обучена. Точность: 82.12%
Модель 'K-ближайших соседей (KNN)' обучена. Точность: 81.01%


In [50]:
print("="*45)
print("ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ (РЕЙТИНГ):")
print("="*45)

# Сортируем модели по убыванию точности
sorted_results = sorted(results.items(), key=lambda item: item[1], reverse=True)

# Выводим красивый рейтинг
for rank, (name, acc) in enumerate(sorted_results, 1):
    print(f"{rank} место: {name} — Точность: {acc * 100:.2f}%")

# Определяем победителя
best_model_name = sorted_results[0][0]
best_model_acc = sorted_results[0][1]

print("-" * 45)
print("ВЫВОД:")
print("После проведения всех этапов Preprocessing (очистка, заполнение пропусков,")
print("кодирование и нормализация) данные стали пригодны для машинного обучения.")
print(f"\nПри сравнении 3-х алгоритмов классификации лучший результат")
print(f"на тестовых данных показала модель: '{best_model_name}'.")
print(f"Она способна предсказывать выживаемость с точностью {best_model_acc * 100:.2f}%.")

ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ (РЕЙТИНГ):
1 место: Случайный лес (Random Forest) — Точность: 82.12%
2 место: K-ближайших соседей (KNN) — Точность: 81.01%
3 место: Логистическая регрессия — Точность: 79.89%
---------------------------------------------
ВЫВОД:
После проведения всех этапов Preprocessing (очистка, заполнение пропусков,
кодирование и нормализация) данные стали пригодны для машинного обучения.

При сравнении 3-х алгоритмов классификации лучший результат
на тестовых данных показала модель: 'Случайный лес (Random Forest)'.
Она способна предсказывать выживаемость с точностью 82.12%.
